In [1]:
import torch
import time
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

model_name = "Qwen/Qwen3-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(model_name)

model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

model.eval()

/home/mv/miniconda3/envs/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 625/625 [00:00<00:00, 1074.34it/s]


Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [2]:
image_path = "sample_car.jpg"

image = Image.open(image_path).convert("RGB")
print("image size:", image.size)

image size: (640, 480)


In [3]:
def make_inputs(messages):
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    print("input_ids:", inputs["input_ids"].shape)

    if "pixel_values" in inputs:
        print("pixel_values:", inputs["pixel_values"].shape)

    if "image_grid_thw" in inputs:
        print("image_grid_thw:", inputs["image_grid_thw"])

    return inputs

In [4]:
messages_text = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe what a car is."}
        ],
    }
]

inputs_text = make_inputs(messages_text)

input_ids: torch.Size([1, 14])


In [5]:
messages_image_short = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": "Describe this image in one sentence."},
        ],
    }
]

inputs_image_short = make_inputs(messages_image_short)

input_ids: torch.Size([1, 317])
pixel_values: torch.Size([1200, 1536])
image_grid_thw: tensor([[ 1, 30, 40]], device='cuda:0')


In [6]:
messages_image_long = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": "Describe this image in detail with many observations."},
        ],
    }
]

inputs_image_long = make_inputs(messages_image_long)

input_ids: torch.Size([1, 319])
pixel_values: torch.Size([1200, 1536])
image_grid_thw: tensor([[ 1, 30, 40]], device='cuda:0')


In [7]:
def measure_prefill(inputs):
    torch.cuda.synchronize()

    with torch.inference_mode():
        t0 = time.perf_counter()

        out = model(
            **inputs,
            use_cache=True,
        )

        torch.cuda.synchronize()
        prefill_time = time.perf_counter() - t0

    print("prefill time:", prefill_time)
    print("cache length:", out.past_key_values.get_seq_length())

    return out, prefill_time

In [8]:
out_text, t_text = measure_prefill(inputs_text)
out_img_short, t_img_short = measure_prefill(inputs_image_short)
out_img_long, t_img_long = measure_prefill(inputs_image_long)

prefill time: 0.23115961800795048
cache length: 14
prefill time: 0.1589127240004018
cache length: 317
prefill time: 0.055360552010824904
cache length: 319


In [9]:
print("text only prefill:", t_text)
print("image + short prompt prefill:", t_img_short)
print("image + long prompt prefill:", t_img_long)

text only prefill: 0.23115961800795048
image + short prompt prefill: 0.1589127240004018
image + long prompt prefill: 0.055360552010824904


In [10]:
messages_two_images = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "image", "image": image_path},
            {"type": "text", "text": "Compare these two images."},
        ],
    }
]

inputs_two_images = make_inputs(messages_two_images)
out_two_images, t_two_images = measure_prefill(inputs_two_images)

print("two images prefill:", t_two_images)

input_ids: torch.Size([1, 617])
pixel_values: torch.Size([2400, 1536])
image_grid_thw: tensor([[ 1, 30, 40],
        [ 1, 30, 40]], device='cuda:0')
prefill time: 0.09896751699852757
cache length: 617
two images prefill: 0.09896751699852757
